In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

len(documents)

72

In [3]:
#Q2

from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [4]:
question = "How does the agentic loop keep calling the model until it stops?"
index.search(
    question,
)

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [5]:
from rag_helper_homework import RAGHomework
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI

openai_client = openai_client = OpenAI()

In [6]:
assistant = RAGHomework(
    index,
    openai_client
)

answer, input_tokens = assistant.rag("How does the agentic loop keep calling the model until it stops?")

print(answer)
print(input_tokens)

It keeps calling the model inside a `while True` loop. After each model response, the code checks whether the response included any `function_call` items:

- if yes, it runs the tool, appends the tool output to `messages`, and loops again
- if no, it breaks out of the loop

So the stop condition is: **no function calls in the current response**.
7136


In [7]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [8]:
len(chunks)

295

In [9]:
index2 = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index2.fit(chunks)

assistant2 = RAGHomework(
    index2,
    openai_client
)

In [10]:
answer2, input_tokens2 = assistant2.rag("How does the agentic loop keep calling the model until it stops?")

print(answer2)
print(input_tokens2)

The loop keeps calling the model in a `while True` loop, and after each turn it checks whether the model returned any `function_call` items.

- If there are function calls, the code runs them, appends the results to `messages`, and loops again.
- If there are no function calls in that turn, `has_function_calls` stays `False`, and the loop `break`s.

So the stop condition is: **no function calls this turn means the model is done**.
2319


In [13]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [14]:
def search(query: str) -> dict[str, str]:
    """
    Search the lessons to answer questions
    """
    return index.search(
        query,
        num_results=5,
    )

agent_tools = Tools()
agent_tools.add_tool(search)

In [16]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the lessons to answer questions',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [19]:
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
""".strip()

In [20]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [21]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received
